# A Practical Guide to Quantitative Finance Interviews — Chapter 4 Probability Theory

**Xinfeng Zhou ("the Green Book")** — worked solutions in code.

Third notebook in the set, alongside the Chapter 2 brain teasers
([`quant_finance_interviews.ipynb`](quant_finance_interviews.ipynb)) and the Chapter 3 calculus
notes ([`quant_finance_calculus.ipynb`](quant_finance_calculus.ipynb)). Probability problems
usually *do* carry a parameter to generalise, so — like Chapter 2 — each gets a short
explanation plus **one general-purpose function** (solving for all `n`), with a Monte-Carlo
check where a closed form is subtle.

Covered so far:

| § | Topic |
|---|-------|
| **4.1** | Basic Probability Definitions & Set Operations |
| **4.2** | Combinatorial Analysis |

## 4.1 Basic Probability Definitions and Set Operations

**The vocabulary.** A random experiment has a **sample space** $\Omega$ (all possible
outcomes); an **event** $A\subseteq\Omega$ is a set of outcomes. A probability measure $P$
assigns each event a number obeying **Kolmogorov's axioms**:
1. $P(A)\ge0$;
2. $P(\Omega)=1$;
3. for mutually exclusive events, $P(A_1\cup A_2\cup\cdots)=\sum_i P(A_i)$.

**Set operations on events** (the algebra of "and / or / not"):
- **Union** $A\cup B$ — $A$ *or* $B$ occurs;
- **Intersection** $A\cap B$ — $A$ *and* $B$ occur;
- **Complement** $A^{c}$ — $A$ does *not* occur, with $P(A^{c})=1-P(A)$;
- **De Morgan:** $(A\cup B)^{c}=A^{c}\cap B^{c}$ and $(A\cap B)^{c}=A^{c}\cup B^{c}$.

**Inclusion–exclusion** repairs the double-counting in a union:
$$P(A\cup B)=P(A)+P(B)-P(A\cap B),$$
$$P(A\cup B\cup C)=P(A)+P(B)+P(C)-P(A\cap B)-P(A\cap C)-P(B\cap C)+P(A\cap B\cap C).$$

**Conditional probability & independence.**
$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}\ \ (P(B)>0),\qquad
A,B\ \text{independent}\iff P(A\cap B)=P(A)\,P(B).$$
Two more workhorses follow: the **law of total probability**
$P(A)=\sum_i P(A\mid B_i)P(B_i)$ over a partition $\{B_i\}$, and **Bayes' rule**
$P(B_i\mid A)=\dfrac{P(A\mid B_i)P(B_i)}{\sum_j P(A\mid B_j)P(B_j)}$.

**Two recurring tricks**, used all over this section:
- **Symmetry** — if two events are interchangeable by relabelling, they carry equal
  probability (so when they also split the remaining "no-tie" cases, each is half of it).
- **Complementary counting** — often $P(A)=1-P(A^{c})$ is far easier than attacking $P(A)$ head-on.

The four problems below are pure applications of these basics.

### 4.1.1 Coin toss game

**Problem.** Gambler $A$ flips $n+1$ fair coins and gambler $B$ flips $n$. What is the
probability that $A$ gets **strictly more heads** than $B$?

**Logic (symmetry — essentially no computation).** Let $H_A,H_B$ be the head counts and
$T_A,T_B$ the tail counts, so $H_A+T_A=n+1$ and $H_B+T_B=n$. Consider
$$X:\ H_A>H_B\ \text{("A has more heads")},\qquad Y:\ T_A>T_B\ \text{("A has more tails")}.$$
- **They are mutually exclusive.** If both held, then $(H_A-H_B)+(T_A-T_B)=(n+1)-n=1$ with
  each difference $\ge1$ — impossible (the two would sum to $\ge2$).
- **One of them always holds.** If neither did, then $H_A\le H_B$ *and* $T_A\le T_B$, forcing
  $n+1=H_A+T_A\le H_B+T_B=n$ — impossible.

So $X$ and $Y$ **partition** the sample space: $P(X)+P(Y)=1$. And since the coins are fair,
swapping heads$\leftrightarrow$tails is a symmetry carrying $X$ to $Y$, so $P(X)=P(Y)$. Hence
$$\boxed{\,P(H_A>H_B)=\tfrac12\,}\qquad\text{for every }n.$$
The function below confirms it by summing the exact binomial probabilities.

In [1]:
from math import comb
from fractions import Fraction

def coin_toss_A_more_heads(n):
    """P(A's heads > B's heads) when A flips n+1 fair coins and B flips n, computed
    exactly from the binomial pmfs. The symmetry argument proves it is 1/2 for every n."""
    total = sum(comb(n + 1, a) * comb(n, b)
                for a in range(n + 2) for b in range(n + 1) if a > b)
    return Fraction(total, 2 ** (2 * n + 1))     # divide by 2^{(n+1)+n}

for n in [0, 1, 2, 5, 10, 25]:
    print(f"n={n:>2}: P(A has more heads) = {coin_toss_A_more_heads(n)}")

n= 0: P(A has more heads) = 1/2
n= 1: P(A has more heads) = 1/2
n= 2: P(A has more heads) = 1/2
n= 5: P(A has more heads) = 1/2
n=10: P(A has more heads) = 1/2
n=25: P(A has more heads) = 1/2


### 4.1.2 Card game

**Problem.** A $52$-card deck has $13$ values ($2,3,\dots,10,J,Q,K,A$), four cards each. You
draw one card and the dealer draws another **without replacement**. You win only if your value
is **strictly higher**; on a tie or a lower value the house wins. What is your winning
probability?

**Logic (symmetry + a single tie term).** Your card and the dealer's are exchangeable, so by
symmetry $P(\text{you}>\text{dealer})=P(\text{dealer}>\text{you})$. Together with a **tie**
these exhaust the outcomes, so
$$P(\text{win})=\frac{1-P(\text{tie})}{2}.$$
Only the tie needs counting: after you draw a card, $51$ remain and $3$ of them share your
value, so $P(\text{tie})=\dfrac{3}{51}=\dfrac1{17}$. Therefore
$$P(\text{win})=\frac{1-\frac1{17}}{2}=\frac{16/17}{2}=\boxed{\dfrac{8}{17}}\approx0.4706,$$
just below a coin flip — the house keeps its edge. In general, for $v$ values with $k$ copies
each ($N=vk$ cards), $P(\text{tie})=\dfrac{k-1}{N-1}$ and
$P(\text{win})=\dfrac12\!\left(1-\dfrac{k-1}{N-1}\right)$.

In [2]:
from fractions import Fraction

def card_win_prob(values=13, copies=4):
    """P(your card's value > dealer's) drawing two cards without replacement from a deck of
    `values` distinct values, `copies` each. P(tie)=(copies-1)/(N-1) and, by symmetry,
    P(win)=(1-P(tie))/2. Returns (P(win), P(tie))."""
    N = values * copies
    p_tie = Fraction(copies - 1, N - 1)
    return (1 - p_tie) / 2, p_tie

w, t = card_win_prob(13, 4)
print(f"standard deck (13 values x 4): P(win) = {w} = {float(w):.4f},  P(tie) = {t}")
for v, k in [(13, 1), (5, 10), (13, 4)]:
    w, t = card_win_prob(v, k)
    print(f"{v:>2} values x {k:>2}: P(win) = {w} = {float(w):.4f}")

standard deck (13 values x 4): P(win) = 8/17 = 0.4706,  P(tie) = 1/17
13 values x  1: P(win) = 1/2 = 0.5000
 5 values x 10: P(win) = 20/49 = 0.4082
13 values x  4: P(win) = 8/17 = 0.4706


### 4.1.3 Drunk passenger

**Problem.** $100$ passengers board in order; passenger $n$ owns seat $n$. The **first**
passenger is drunk and takes a **uniformly random** seat. Everyone after sits in their own seat
if it is free, otherwise in a uniformly random free seat. What is the probability **you**
(passenger $100$) end up in your own seat?

**Logic (a clean symmetry).** Follow the "displaced" passenger — initially the drunk one. Each
time a displaced passenger picks at random, the process ends the moment someone sits in **seat
$1$** (then every later passenger, you included, finds their own seat free → you **win**) or in
**seat $100$** (then you are ultimately bumped → you **lose**). Any random pick that hits some
*other* seat merely hands the "displaced" role to a new passenger and continues. So the outcome
is a race between seat $1$ and seat $100$, and at each random choice those two seats are
**equally likely** to be taken — so by symmetry the process ends on each with probability
$\tfrac12$:
$$\boxed{\,P(\text{you get seat }100)=\tfrac12\,}\qquad(n\ge2).$$
Strikingly this is $\tfrac12$ for **any** number of passengers $\ge2$ — the $100$ is a red
herring. Exact value and a Monte-Carlo check below.

In [3]:
import random
from fractions import Fraction

def drunk_passenger_exact(n):
    """P(passenger n gets their own seat) in the drunk-first-passenger problem:
    exactly 1/2 for n >= 2 (and 1 for n = 1)."""
    return Fraction(1) if n <= 1 else Fraction(1, 2)

def drunk_passenger_sim(n, trials=20000, seed=0):
    """Monte-Carlo estimate of the same probability, boarding the plane `trials` times."""
    rng = random.Random(seed)
    wins = 0
    for _ in range(trials):
        taken = bytearray(n + 1)                       # seats 1..n (index 0 unused)
        taken[rng.randint(1, n)] = 1                    # drunk passenger 1
        for i in range(2, n):                           # sober passengers 2..n-1
            if not taken[i]:
                taken[i] = 1                            # own seat free -> take it
            else:
                free = [s for s in range(1, n + 1) if not taken[s]]
                taken[rng.choice(free)] = 1             # else a random free seat
        wins += (taken[n] == 0)                          # you win iff seat n is still free
    return wins / trials

for n in [2, 5, 100]:
    print(f"n={n:>3}: exact = {drunk_passenger_exact(n)}   simulated = {drunk_passenger_sim(n):.3f}")

n=  2: exact = 1/2   simulated = 0.503
n=  5: exact = 1/2   simulated = 0.495


n=100: exact = 1/2   simulated = 0.503


### 4.1.4 N points on a circle

**Problem.** $N$ points are dropped uniformly at random on a circle's circumference. What is the
probability they **all lie within some semicircle**?

**Logic (fix a point, then sum disjoint cases).** For each point $i$, let $E_i$ be the event
"starting at point $i$ and sweeping **clockwise**, the next half-circle contains all the other
$N-1$ points." Each other point independently falls in that fixed semicircle with probability
$\tfrac12$, so
$$P(E_i)=\left(\tfrac12\right)^{N-1}.$$
The events $E_1,\dots,E_N$ are **mutually exclusive** — at most one point can be the clockwise
"first" point of an all-containing semicircle. Since "all in some semicircle" is exactly "some
$E_i$ occurs,"
$$P(\text{all within a semicircle})=\sum_{i=1}^{N}P(E_i)=\boxed{\dfrac{N}{2^{\,N-1}}}.$$
Sanity: $N=2\Rightarrow1$ (two points always fit), $N=3\Rightarrow\tfrac34$,
$N=4\Rightarrow\tfrac12$. The function returns the exact value; the simulation checks it via the
equivalent test *"the largest gap between adjacent points is at least half the circle."*

In [4]:
import random
from fractions import Fraction

def semicircle_prob(N):
    """P(all N uniformly-random points on a circle lie within some semicircle) = N / 2^{N-1}."""
    return Fraction(N, 2 ** (N - 1))

def semicircle_sim(N, trials=200000, seed=0):
    """Monte-Carlo check: all points fit in a semicircle iff some gap between adjacent points
    (around the circle) is at least half the circumference."""
    rng = random.Random(seed)
    hits = 0
    for _ in range(trials):
        pts = sorted(rng.random() for _ in range(N))        # positions as fractions of the circle
        gaps = [pts[i + 1] - pts[i] for i in range(N - 1)] + [1 - pts[-1] + pts[0]]
        hits += (max(gaps) >= 0.5)
    return hits / trials

for N in [2, 3, 4, 5, 6]:
    exact = semicircle_prob(N)
    print(f"N={N}: exact = {exact} = {float(exact):.4f}   simulated = {semicircle_sim(N):.4f}")

N=2: exact = 1 = 1.0000   simulated = 1.0000


N=3: exact = 3/4 = 0.7500   simulated = 0.7506


N=4: exact = 1/2 = 0.5000   simulated = 0.5011


N=5: exact = 5/16 = 0.3125   simulated = 0.3131


N=6: exact = 3/16 = 0.1875   simulated = 0.1882


## 4.2 Combinatorial Analysis

Counting is the engine of discrete probability: most "what's the chance" questions reduce to
*(favourable arrangements) / (total arrangements)*. Five tools do almost all the work.

**Basic principle of counting (multiplication rule).** If a task is carried out in $k$
independent stages offering $n_1,n_2,\dots,n_k$ choices, the number of overall outcomes is
$$n_1\times n_2\times\cdots\times n_k .$$
(For *mutually exclusive alternatives* you **add** rather than multiply.)

**Permutations — ordered arrangements.** The number of ways to arrange $r$ of $n$ distinct
objects, where **order matters**, is
$$P(n,r)=\frac{n!}{(n-r)!};\qquad\text{all }n\text{ objects: }P(n,n)=n! .$$

**Combinations — unordered selections.** The number of ways to choose $r$ of $n$ objects, where
**order does not matter**, is
$$\binom{n}{r}=\frac{n!}{r!\,(n-r)!}=\frac{P(n,r)}{r!},\qquad \binom{n}{r}=\binom{n}{n-r}.$$

**Binomial theorem.** Those combinations are exactly the coefficients of
$$(x+y)^{n}=\sum_{k=0}^{n}\binom{n}{k}x^{k}y^{n-k},$$
so putting $x=y=1$ gives $\displaystyle\sum_{k=0}^{n}\binom{n}{k}=2^{n}$ — the number of subsets
of an $n$-element set.

**Inclusion–exclusion principle.** To count a union without double-counting overlaps,
$$\Big|\bigcup_{i=1}^{n}A_i\Big|=\sum_i|A_i|-\sum_{i<j}|A_i\cap A_j|+\sum_{i<j<k}|A_i\cap A_j\cap A_k|-\cdots+(-1)^{\,n+1}\Big|\bigcap_{i=1}^{n}A_i\Big| .$$
(The derangement count in 4.2.5 is a textbook application.)

The five problems below apply these directly.

### 4.2.1 Poker hands

**Problem.** From a standard $52$-card deck ($13$ values $\times$ $4$ suits), a poker hand is
$5$ cards. Find the probability of **four-of-a-kind**, a **full house** (three of one value and
two of another), and **two pairs**.

**Counting.** Every hand is equally likely, so each probability is
(favourable hands)$/\binom{52}{5}$ with $\binom{52}{5}=2{,}598{,}960$. Build each favourable hand
by the multiplication rule:

- **Four-of-a-kind:** pick the quad's value ($13$), take all $4$ suits ($\binom{4}{4}=1$), then
  any $5$th card from the other $48$: $\;13\cdot1\cdot48=624$.
- **Full house:** pick the triple's value ($13$) and $3$ of its suits ($\binom{4}{3}=4$), then the
  pair's value ($12$) and $2$ suits ($\binom{4}{2}=6$): $\;13\cdot4\cdot12\cdot6=3{,}744$.
- **Two pairs:** pick the two pair-values ($\binom{13}{2}=78$), $2$ suits for each
  ($\binom{4}{2}^{2}=36$), then a $5$th card of a different value ($11\cdot4=44$):
  $\;78\cdot36\cdot44=123{,}552$.

So $P(\text{four})\approx0.00024$, $P(\text{full house})\approx0.00144$,
$P(\text{two pairs})\approx0.0475$. The function computes these, generalised to any
values$\times$suits deck.

In [5]:
from math import comb

def poker_probabilities(values=13, suits=4):
    """Counts of four-of-a-kind, full house, and two pairs in a 5-card hand from a deck of
    `values` distinct values x `suits` suits; each probability is count / C(values*suits, 5).
    All counts come from the multiplication rule."""
    total = comb(values * suits, 5)
    four = values * comb(suits, 4) * (values - 1) * suits
    full = values * comb(suits, 3) * (values - 1) * comb(suits, 2)
    two_pair = comb(values, 2) * comb(suits, 2) ** 2 * (values - 2) * suits
    return {"four_of_a_kind": four, "full_house": full, "two_pairs": two_pair, "total": total}

r = poker_probabilities()
for hand in ("four_of_a_kind", "full_house", "two_pairs"):
    c = r[hand]
    print(f"{hand:>15}: {c:>7,} / {r['total']:,} = {c/r['total']:.6f}  (~1 in {r['total']/c:,.0f})")

 four_of_a_kind:     624 / 2,598,960 = 0.000240  (~1 in 4,165)
     full_house:   3,744 / 2,598,960 = 0.001441  (~1 in 694)
      two_pairs: 123,552 / 2,598,960 = 0.047539  (~1 in 21)


### 4.2.2 Hopping rabbit

**Problem.** A rabbit at the bottom of a staircase of $n$ stairs hops up $1$ or $2$ stairs at a
time. How many distinct ways can it reach the top?

**Logic (recursion → Fibonacci).** Let $f(n)$ be the number of ways. The rabbit's **last** hop
lands on stair $n$ from either stair $n-1$ (a $1$-hop) or stair $n-2$ (a $2$-hop), and those two
sets of routes are disjoint, so
$$f(n)=f(n-1)+f(n-2),\qquad f(1)=1,\ f(2)=2.$$
That is the **Fibonacci** recurrence: $f(n)=1,2,3,5,8,13,\dots$ (indeed $f(n)=F_{n+1}$). The
function returns $f(n)$ in $O(n)$.

In [6]:
def rabbit_ways(n):
    """Ways to climb n stairs in 1- or 2-hops. f(n)=f(n-1)+f(n-2) (the last hop comes from stair
    n-1 or n-2), with f(0)=f(1)=1 -> a Fibonacci sequence."""
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print("stairs n :", list(range(1, 11)))
print("ways f(n):", [rabbit_ways(n) for n in range(1, 11)])

stairs n : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
ways f(n): [1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### 4.2.3 Screwy pirates 2

**Problem.** $11$ pirates lock the loot in a safe that opens only when **any majority**
($\ge6$) work together. Each lock can carry many keys, each key opens one lock, and a pirate may
hold several keys. What is the **smallest number of locks**, and how many **keys per pirate**?

**Logic (block every minority).** The requirement is: every group of $6$ opens *all* locks, yet
every group of $5$ is stopped by *some* lock. So dedicate one lock to each minority of size $5$
and give that lock's keys only to the **other $6$** pirates.

- A group of $5$ is exactly one such minority $S$; the lock assigned to $S$ is held only by the
  $6$ outsiders, so this group lacks it — blocked. ✓
- A group of $6$: for *any* lock (assigned to some $5$-set $S$), the six include at least one
  pirate outside $S$, who holds that key — so they open everything. ✓

Hence **one lock per $5$-subset**: $\binom{11}{5}=462$ locks. A given pirate holds a lock's key
iff he is *not* in its $5$-subset, i.e. the subset is chosen from the other $10$:
$\binom{10}{5}=252$ keys each. In general, for $n$ pirates with majority threshold $m$ (minority
$m-1$): $\binom{n}{m-1}$ locks and $\binom{n-1}{m-1}$ keys per pirate.

In [7]:
from math import comb

def screwy_pirates2(n=11, majority=6):
    """Smallest number of locks and keys-per-pirate so any `majority` pirates can open the safe
    but no smaller group can. One lock per minority of size majority-1, its keys held by everyone
    outside that minority. Returns (locks, keys_per_pirate)."""
    minority = majority - 1
    return comb(n, minority), comb(n - 1, minority)

print("11 pirates, majority 6 ->", screwy_pirates2(11, 6), "(locks, keys per pirate)")
for n, m in [(3, 2), (5, 3), (11, 6)]:
    print(f"  n={n:>2}, majority={m}: {screwy_pirates2(n, m)}")

11 pirates, majority 6 -> (462, 252) (locks, keys per pirate)
  n= 3, majority=2: (3, 2)
  n= 5, majority=3: (10, 6)
  n=11, majority=6: (462, 252)


### 4.2.4 Chess tournament

**Problem.** $2^{n}$ players with strictly ranked skills play a knockout tournament; opponents
are drawn **at random** each round (except the final), and the stronger player always wins. What
is the probability that the top two, players $1$ and $2$, meet in the **final**?

**Logic.** Player $1$ beats everyone, so player $1$ reaches the final with certainty. Players $1$
and $2$ therefore meet in the final **iff player $2$ survives every earlier round**, i.e. iff
player $2$ is never drawn against player $1$ before the final. In a round beginning with $m$
players, player $2$ faces a uniformly random one of the other $m-1$, avoiding player $1$ with
probability $\tfrac{m-2}{m-1}$. Multiplying over the pre-final rounds ($m=2^{n},2^{n-1},\dots,4$),
$$P=\prod_{m\in\{4,8,\dots,2^{n}\}}\frac{m-2}{m-1}=\boxed{\dfrac{2^{\,n-1}}{2^{\,n}-1}} .$$
(Check: $n=2\Rightarrow\tfrac23$, $n=3\Rightarrow\tfrac47$; as $n$ grows it tends to $\tfrac12$.)

In [8]:
from fractions import Fraction

def chess_final_prob(n):
    """P(the two best of 2^n players meet in the final of a random knockout). Player 1 always
    wins; player 2 must dodge player 1 in each round before the final. The product telescopes
    to 2^{n-1}/(2^n - 1)."""
    prob, m = Fraction(1), 2 ** n
    while m > 2:
        prob *= Fraction(m - 2, m - 1)        # player 2 avoids player 1 this round
        m //= 2
    return prob

for n in range(1, 7):
    p = chess_final_prob(n)
    print(f"n={n}: {2**n:>2} players -> P = {p} = {float(p):.4f}   [2^(n-1)/(2^n-1)]")

n=1:  2 players -> P = 1 = 1.0000   [2^(n-1)/(2^n-1)]
n=2:  4 players -> P = 2/3 = 0.6667   [2^(n-1)/(2^n-1)]
n=3:  8 players -> P = 4/7 = 0.5714   [2^(n-1)/(2^n-1)]
n=4: 16 players -> P = 8/15 = 0.5333   [2^(n-1)/(2^n-1)]
n=5: 32 players -> P = 16/31 = 0.5161   [2^(n-1)/(2^n-1)]
n=6: 64 players -> P = 32/63 = 0.5079   [2^(n-1)/(2^n-1)]


### 4.2.5 Application letters

**Problem.** Five personalised cover letters are stuffed into five addressed envelopes **at
random**. What is the probability that **all five** go to the wrong firm?

**Logic (derangements via inclusion–exclusion).** "All wrong" means the random permutation of
letters leaves **no letter in its correct envelope** — a *derangement*. Let $A_i$ be the event
that letter $i$ *is* correctly placed; inclusion–exclusion on $\bigcup A_i$ (at least one
correct) leaves, for the complement (none correct),
$$P(\text{all wrong})=\frac{D_n}{n!}=\sum_{k=0}^{n}\frac{(-1)^{k}}{k!},$$
where $D_n$ is the number of derangements. For $n=5$,
$$P=1-1+\tfrac12-\tfrac16+\tfrac1{24}-\tfrac1{120}=\frac{44}{120}=\boxed{\frac{11}{30}}\approx0.3667 .$$
As $n\to\infty$ this converges rapidly to $1/e\approx0.3679$ — it is essentially $1/e$ already at
$n=5$.

In [9]:
from math import factorial
from fractions import Fraction

def derangement_prob(n):
    """P(a random permutation of n items leaves nothing in its own place) = D_n/n!
    = sum_{k=0}^n (-1)^k/k!  ->  1/e as n grows."""
    return sum(Fraction((-1) ** k, factorial(k)) for k in range(n + 1))

for n in [3, 4, 5, 6, 10]:
    p = derangement_prob(n)
    print(f"n={n:>2}: P(all wrong) = {p} = {float(p):.5f}   (D_n = {round(factorial(n) * float(p))})")
print(f"limit 1/e = {1 / 2.718281828459045:.5f}")

n= 3: P(all wrong) = 1/3 = 0.33333   (D_n = 2)
n= 4: P(all wrong) = 3/8 = 0.37500   (D_n = 9)
n= 5: P(all wrong) = 11/30 = 0.36667   (D_n = 44)
n= 6: P(all wrong) = 53/144 = 0.36806   (D_n = 265)
n=10: P(all wrong) = 16481/44800 = 0.36788   (D_n = 1334961)
limit 1/e = 0.36788


---
*More of Chapter 4 as I keep reading: conditional probability and the standard distributions.*